# Chapter 20
## Chemical Synapses
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter20.ipynb)

## About this chapter

Chemical synapses turn presynaptic activity into a conductance-based
current $I_{\rm syn}=g_{\rm syn}s(v_{\rm syn}-v)$ on the postsynaptic
cell. A release variable $q$ can rise rapidly after a spike and feed the
gate $s$, which decays more slowly -- the rise and decay time constants
shape both the peak time and duration of the conductance. `B_JAHR_STEVENS`
plots the NMDA magnesium-block factor; the `RTM_PLOT_*` examples build up
the release-and-gate synapse model on a single RTM neuron, including
solving for the release time constant that gives a prescribed peak time;
`S_BUILDUP`/`S_SLOW_BUILDUP` show how closely spaced spikes build up
(and, with a slow decay, retain) synaptic activation between events;
`RTM_WITH_AUTAPSE_F_I_CURVE` closes an excitatory self-synapse (autapse)
onto the same RTM neuron and sweeps its forward/backward F-I curve --
its inner time-stepping loop is JIT-compiled with numba, since the
uncompiled sweep took several minutes each direction.

See [`README.md`](chapter20.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import math
import numpy as np
from numpy import exp
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit
from mnd.core import alpha_h, alpha_m, alpha_n, beta_h, beta_m, beta_n, h_inf, m_inf, n_inf

## NMDA Magnesium-Block Factor

The Jahr-Stevens voltage-dependent factor $B(v)$ that relieves magnesium
block of the NMDA channel as the postsynaptic cell depolarizes.

In [ ]:
def simulate_b_jahr_stevens(v=None):
    if v is None:
        v = np.arange(-100, 51)
    B = 1.0 / (1 + np.exp(-0.062 * v) / 3.57)
    return v, B


def plot_b_jahr_stevens(v, B):
    plt.figure(figsize=(7, 3.5))
    plt.plot(v, B)
    plt.xlabel(r'$v_{\rm post}$')
    plt.ylabel('$B$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_b_jahr_stevens(*simulate_b_jahr_stevens())

## RTM with a Single-Variable Synaptic Gate

A single gate $s$ rises directly from presynaptic voltage and decays with
$\tau_d$: $\dot s=\tfrac12(1+\tanh(v/10))(1-s)/\tau_r-s/\tau_d$.
Comparing a fast ($\tau_r=0.2$) and slow ($\tau_r=1.0$) rise shows how
$\tau_r$ shapes the gate's rise time without a separate release variable.

In [ ]:
def simulate_rtm_plot_s(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0,
                         i_ext=1.0, tau_r=0.2, tau_d=2.0, t_final=100.0, dt=0.01, v0=-70.0):
    def derivative(x0, t):
        v, n, h, s = x0
        m = m_inf(v)
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l))
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        ds = 0.5 * (1.0 + np.tanh(0.1 * v)) * (1 - s) / tau_r - s / tau_d
        return [dv, dn, dh, ds]

    x0 = [v0, n_inf(v0), h_inf(v0), 0.0]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0], sol[:, -1]


def plot_rtm_plot_s(t, V, S1, S2):
    fig, ax = plt.subplots(3, figsize=(7, 5), sharex=True)
    ax[0].plot(t, V, lw=2, c="k")
    ax[1].plot(t, S1, lw=2, c="k")
    ax[2].plot(t, S2, lw=2, c="k")

    ax[0].set_xlim(min(t), max(t))
    ax[0].set_ylim(-100, 100)
    ax[1].set_ylim([0, 1])
    ax[2].set_ylim([0, 1])
    ax[2].set_xlabel("time [ms]", fontsize=14)
    ax[0].set_ylabel("v [mV]", fontsize=14)
    ax[1].set_ylabel("s", fontsize=14)
    ax[2].set_ylabel("s", fontsize=14)
    ax[0].set_yticks([-100, 0, 100])
    ax[1].set_yticks([0, 0.5, 1])
    ax[2].set_yticks([0, 0.5, 1])
    plt.tight_layout()
    plt.show()

In [ ]:
t, V, S1 = simulate_rtm_plot_s(tau_r=0.2, tau_d=2.0)
_, _, S2 = simulate_rtm_plot_s(tau_r=1.0, tau_d=2.0)
plot_rtm_plot_s(t, V, S1, S2)

## RTM with Release and Gate Variables

Separating transmitter release $q$ from the gate $s$
($\dot q=\tfrac12(1+\tanh(v/10))(1-q)/\tau_r-q/\tau_d$,
$\dot s=q(1-s)/\tau_r-s/\tau_d$) delays and smooths $s$ relative to the
single-variable gate above -- a two-stage synapse has a delayed profile
that a direct gate cannot reproduce.

In [ ]:
def simulate_rtm_plot_q(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0,
                         i_ext=1.0, tau_r=0.1, tau_d=2.0, t_final=100.0, dt=0.01, v0=-70.0):
    def derivative(x0, t):
        v, n, h, q, s = x0
        m = m_inf(v)
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l))
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        dq = 0.5 * (1.0 + np.tanh(0.1 * v)) * (1 - q) / tau_r - q / tau_d
        ds = q * (1.0 - s) / tau_r - s / tau_d
        return [dv, dn, dh, dq, ds]

    x0 = [v0, n_inf(v0), h_inf(v0), 0.0, 0.0]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0], sol[:, 3], sol[:, 4]


def plot_rtm_plot_q(t, V, Q, S):
    fig, ax = plt.subplots(2, figsize=(7, 5), sharex=True)
    ax[0].plot(t, V, lw=2, c="k")
    ax[1].plot(t, S, lw=1, c="b", ls="--", label="s")
    ax[1].plot(t, Q, lw=2, c="r", label="q")

    ax[0].set_xlim(min(t), max(t))
    ax[0].set_ylim(-100, 50)
    ax[1].set_ylim([0, 1])
    ax[1].set_xlabel("time [ms]", fontsize=14)
    ax[0].set_ylabel("v [mV]", fontsize=14)
    ax[1].set_ylabel("q", fontsize=14)
    ax[0].set_yticks([-100, 0, 50])
    ax[1].set_yticks([0, 0.5, 1])
    ax[1].legend(frameon=False)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_rtm_plot_q(*simulate_rtm_plot_q())

## RTM with a Prescribable Release Time Constant (shared by the examples below)

A release variable driven at a fixed rate
($\dot q=5(1+\tanh(v/10))(1-q)-q/\tau_{d,q}$) whose decay $\tau_{d,q}$ is
solved for numerically (`tau_d_q_function`, bisecting on
`tau_peak_function`) so the gate $s$ peaks at a prescribed time
$\hat\tau$ after a presynaptic spike.

In [ ]:
def _two_stage_derivative(x0, t, i_ext, tau_r, tau_d, tau_d_q,
                           c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                           v_k=-100.0, v_na=50.0, v_l=-67.0):
    v, n, h, q, s = x0
    m = m_inf(v)
    dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
          - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l))
    dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
    dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
    dq = 0.5 * (1.0 + np.tanh(0.1 * v)) * (1 - q) * 10.0 - q / tau_d_q
    ds = q * (1 - s) / tau_r - s / tau_d
    return [dv, dn, dh, dq, ds]


def simulate_two_stage_synapse(tau_r, tau_d, tau_d_q, i_ext=0.12, t_final=2000.0, dt=0.01, v0=-70.0):
    x0 = [v0, n_inf(v0), h_inf(v0), 0.0, 0.0]
    t = np.arange(0, t_final, dt)
    sol = odeint(_two_stage_derivative, x0, t, args=(i_ext, tau_r, tau_d, tau_d_q))
    return t, sol[:, 0], sol[:, 3], sol[:, 4]


def tau_peak_function(tau_d, tau_r, tau_d_q):
    """Time (from a delta-function pulse of transmitter release) at which
    the synaptic gate s peaks, for exponential-rise/decay time constants
    tau_r/tau_d and release time constant tau_d_q."""
    dt = 0.01
    dt05 = dt / 2
    s, t = 0.0, 0.0
    s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05 * s_inc
        s_inc_tmp = exp(-(t + dt05) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt * s_inc_tmp
        t = t + dt
        s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


def tau_d_q_function(tau_d, tau_r, tau_hat):
    """Release time constant tau_d_q so that tau_peak_function reproduces
    the prescribed tau_hat (bisection, since there's no closed form)."""
    tau_d_q_left = 1.0
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2


def plot_two_stage_synapse(t, V, S1, S2, title1, title2):
    fig, ax = plt.subplots(3, figsize=(7, 5), sharex=True)
    ax[0].plot(t, V, lw=2, c="k")
    ax[1].plot(t, S1, lw=2, c="k")
    ax[2].plot(t, S2, lw=2, c="k")

    ax[0].set_xlim(min(t), max(t))
    ax[0].set_ylim(-100, 100)
    ax[1].set_ylim([0, 1])
    ax[2].set_ylim([0, 1])
    ax[2].set_xlabel("time [ms]", fontsize=14)
    ax[0].set_ylabel("v [mV]", fontsize=14)
    ax[1].set_ylabel("s", fontsize=14)
    ax[2].set_ylabel("s", fontsize=14)
    ax[0].set_yticks([-100, 0, 100])
    ax[1].set_yticks([0, 0.5, 1])
    ax[2].set_yticks([0, 0.5, 1])
    ax[1].set_title(title1)
    ax[2].set_title(title2)
    plt.tight_layout()
    plt.show()

## Two-Stage Synapse: Two Timing Choices

Fixed $\tau_{d,q}=10$ vs. $\tau_{d,q}=100$ ms, both with matching $\tau_r$.

In [ ]:
t, V, _, S1 = simulate_two_stage_synapse(tau_r=10.0, tau_d=300.0, tau_d_q=10.0)
_, _, _, S2 = simulate_two_stage_synapse(tau_r=100.0, tau_d=300.0, tau_d_q=100.0)
plot_two_stage_synapse(t, V, S1, S2,
                        r"$\tau_d=300 ms, \tau_r=10 ms, \tau_{d,q}$=10 ms",
                        r"$\tau_d=300 ms, \tau_r=100 ms, \tau_{d,q}$=100 ms")

## Two-Stage Synapse: Prescribed Peak Time

Solves for $\tau_{d,q}$ so the gate peaks at $\hat\tau=20$ ms (fast) and $\hat\tau=150$ ms (slow), instead of fixing $\tau_{d,q}$ directly.

In [ ]:
tau_d_q_1 = tau_d_q_function(tau_d=300.0, tau_r=10.0, tau_hat=20.0)
t, V, _, S1 = simulate_two_stage_synapse(tau_r=10.0, tau_d=300.0, tau_d_q=tau_d_q_1)

tau_d_q_2 = tau_d_q_function(tau_d=300.0, tau_r=100.0, tau_hat=150.0)
_, _, _, S2 = simulate_two_stage_synapse(tau_r=100.0, tau_d=300.0, tau_d_q=tau_d_q_2)

plot_two_stage_synapse(t, V, S1, S2,
                        rf"$\tau_d=300 ms, \tau_r=10 ms, \tau_{{d,q}}$={tau_d_q_1:.3g} ms",
                        rf"$\tau_d=300 ms, \tau_r=100 ms, \tau_{{d,q}}$={tau_d_q_2:.3g} ms")

## Synaptic Buildup from Repeated Spikes

With the neuron spiking repetitively (autonomously, from a suprathreshold
$I$), closely spaced presynaptic events build up the gate $s$ faster than
it decays between spikes. Comparing a fast-decay ($\tau_d=300$) and a
much-slower-decay ($\tau_d=500$, $\tau_r=100$, $\tau_{d,q}=1$) case shows
how a slow decay retains activation between events instead of resetting.

In [ ]:
def simulate_s_buildup(tau_r, tau_d, tau_d_q, i_ext=0.2, t_final=2000.0, dt=0.01, v0=-70.0,
                        spike_threshold=-20.0):
    t, V, Q, S = simulate_two_stage_synapse(tau_r, tau_d, tau_d_q, i_ext=i_ext,
                                             t_final=t_final, dt=dt, v0=v0)
    crossing = np.where((V[:-1] <= spike_threshold) & (V[1:] > spike_threshold))[0]
    t_spikes = ((crossing * dt * (V[crossing] - spike_threshold)
                 + (crossing + 1) * dt * (spike_threshold - V[crossing + 1]))
                / (V[crossing] - V[crossing + 1]))
    period = t_spikes[-1] - t_spikes[-2]
    return t, V, S, period


def plot_s_buildup(t, V, S, period):
    print(f"Period is {period:10.3f} ms")
    fig, ax = plt.subplots(2, figsize=(7, 5), sharex=True)
    ax[0].plot(t, V, lw=2, c="k")
    ax[1].plot(t, S, lw=2, c="k")

    ax[0].set_xlim(min(t), max(t))
    ax[0].set_ylim(-100, 100)
    ax[1].set_xlabel("time [ms]", fontsize=14)
    ax[0].set_ylabel("v [mV]", fontsize=14)
    ax[1].set_ylabel("s", fontsize=14)
    ax[0].set_yticks([-100, 0, 100])
    plt.tight_layout()
    plt.show()

In [ ]:
plot_s_buildup(*simulate_s_buildup(tau_r=10.0, tau_d=300.0, tau_d_q=5.0, i_ext=0.2))

## Synaptic Buildup: Slow Decay

Condition is $\tau_d\gg T$ (period) and $\tau_{d,q}\ll\tau_r$.

In [ ]:
plot_s_buildup(*simulate_s_buildup(tau_r=100.0, tau_d=500.0, tau_d_q=1.0, i_ext=0.2))

## RTM with an Excitatory Autapse: F-I Curve

An RTM neuron synapsing onto itself (an autapse) through the same
release-and-gate synapse as above. Forward and backward sweeps over 31
values of $I$ each integrate to steady state (or 4 spikes), continuing
from the previous $I$'s final state -- the same MATLAB-style hysteresis
scan as chapter17's F-I curves. The inner time-stepping loop is
JIT-compiled with numba: the uncompiled Heun/RK2 sweep took several
minutes per direction at `dt=0.005`, well under a second once compiled.

In [ ]:
@njit
def _autapse_m_inf(v):
    am = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    bm = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return am / (am + bm)


@njit
def _autapse_run_to_frequency(i_ext, v, m, h, n, q, s, c, g_na, g_k, g_l, v_na, v_k, v_l,
                               g_syn, v_syn, tau_r, tau_d, tau_dq, dt, dt05, t_max_steps, N):
    win_maxv = win_minv = v
    win_maxm = win_minm = m
    win_maxh = win_minh = h
    win_maxn = win_minn = n
    win_maxq = win_minq = q
    win_maxs = win_mins = s
    num_spikes = 0
    t3 = 0.0
    t4 = 0.0

    for k in range(1, t_max_steps + 1):
        v_prev = v
        alpha_h_v = 0.128 * math.exp(-(v + 50) / 18)
        alpha_n_v = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
        beta_h_v = 4.0 / (1 + math.exp(-(v + 27) / 5))
        beta_n_v = 0.5 * math.exp(-(v + 57) / 40)

        v_inc = (g_na * math.pow(m, 3.0) * h * (v_na - v) + g_k * math.pow(n, 4.0) * (v_k - v)
                 + g_l * (v_l - v) + g_syn * s * (v_syn - v) + i_ext) / c
        h_inc = alpha_h_v * (1 - h) - beta_h_v * h
        n_inc = alpha_n_v * (1 - n) - beta_n_v * n
        q_inc = 5 * (1 + math.tanh(v / 10)) * (1 - q) - q / tau_dq
        s_inc = q * (1 - s) / tau_r - s / tau_d

        v_tmp = v + dt05 * v_inc
        m_tmp = _autapse_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        q_tmp = q + dt05 * q_inc
        s_tmp = s + dt05 * s_inc

        alpha_h_vtmp = 0.128 * math.exp(-(v_tmp + 50) / 18)
        alpha_n_vtmp = 0.032 * (v_tmp + 52) / (1 - math.exp(-(v_tmp + 52) / 5))
        beta_h_vtmp = 4.0 / (1 + math.exp(-(v_tmp + 27) / 5))
        beta_n_vtmp = 0.5 * math.exp(-(v_tmp + 57) / 40)

        v_inc = (g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp) + g_k * math.pow(n_tmp, 4.0) * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + g_syn * s_tmp * (v_syn - v_tmp) + i_ext) / c
        h_inc = alpha_h_vtmp * (1 - h_tmp) - beta_h_vtmp * h_tmp
        n_inc = alpha_n_vtmp * (1 - n_tmp) - beta_n_vtmp * n_tmp
        q_inc = 5 * (1 + math.tanh(v_tmp / 10)) * (1 - q_tmp) - q_tmp / tau_dq
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v = v + dt * v_inc
        m = _autapse_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc
        q = q + dt * q_inc
        s = s + dt * s_inc

        win_maxv = max(win_maxv, v); win_minv = min(win_minv, v)
        win_maxm = max(win_maxm, m); win_minm = min(win_minm, m)
        win_maxh = max(win_maxh, h); win_minh = min(win_minh, h)
        win_maxn = max(win_maxn, n); win_minn = min(win_minn, n)
        win_maxq = max(win_maxq, q); win_minq = min(win_minq, q)
        win_maxs = max(win_maxs, s); win_mins = min(win_mins, s)

        if v < -20 and v_prev >= -20:
            num_spikes += 1
            ts = (k * dt * (20 + v_prev) + (k - 1) * dt * (-20 - v)) / (v_prev - v)
            if num_spikes == 3:
                t3 = ts
            elif num_spikes == 4:
                t4 = ts
                return 1000.0 / (t4 - t3), v, m, h, n, q, s, 1

        if k % N == 0:
            if (win_maxv - win_minv) < 1e-4 * abs(win_maxv + win_minv) and \
               (win_maxm - win_minm) < 1e-4 * abs(win_maxm + win_minm) and \
               (win_maxh - win_minh) < 1e-4 * abs(win_maxh + win_minh) and \
               (win_maxn - win_minn) < 1e-4 * abs(win_maxn + win_minn) and \
               (win_maxq - win_minq) < 1e-4 * abs(win_maxq + win_minq) and \
               (win_maxs - win_mins) < 1e-4 * abs(win_maxs + win_mins):
                return 0.0, v, m, h, n, q, s, 0
            win_maxv = win_minv = v
            win_maxm = win_minm = m
            win_maxh = win_minh = h
            win_maxn = win_minn = n
            win_maxq = win_minq = q
            win_maxs = win_mins = s

    return 0.0, v, m, h, n, q, s, -1


def simulate_rtm_with_autapse_f_i_curve(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                         v_k=-100.0, v_na=50.0, v_l=-67.0,
                                         g_syn=0.1, v_syn=0.0, tau_d=5.0, tau_r=0.2, tau_peak=0.6,
                                         dt=0.005, i_ext_vec=None):
    dt05 = dt / 2
    N = round(1000 / dt)
    t_max_steps = round(2000.0 / dt)
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)

    def run_to_frequency(i_ext, v, m, h, n, q, s):
        freq, v, m, h, n, q, s, status = _autapse_run_to_frequency(
            i_ext, v, m, h, n, q, s, c, g_na, g_k, g_l, v_na, v_k, v_l,
            g_syn, v_syn, tau_r, tau_d, tau_dq, dt, dt05, t_max_steps, N)
        if status == -1:
            raise RuntimeError(f"did not settle within {t_max_steps} steps at I={i_ext}")
        return freq, v, m, h, n, q, s

    if i_ext_vec is None:
        i_ext_low, i_ext_high = 0.0, 0.15
        i_ext_vec = i_ext_low + np.arange(31) / 30 * (i_ext_high - i_ext_low)

    f_forward = np.zeros(len(i_ext_vec))
    v, m, h, n, q, s = -70.0, _autapse_m_inf(-70.0), 0.7, 0.6, 0.0, 0.0
    for ijk, i_ext in enumerate(i_ext_vec):
        f_forward[ijk], v, m, h, n, q, s = run_to_frequency(i_ext, v, m, h, n, q, s)

    f_backward = np.zeros(len(i_ext_vec))
    for ijk in range(len(i_ext_vec) - 1, -1, -1):
        f_backward[ijk], v, m, h, n, q, s = run_to_frequency(i_ext_vec[ijk], v, m, h, n, q, s)

    ind = np.where(f_forward == 0)[0].max()
    I_c = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2
    ind = np.where(f_backward == 0)[0].max()
    I_star = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2

    return f_forward, f_backward, I_c, I_star, i_ext_vec


def plot_rtm_with_autapse_f_i_curve(f_forward, f_backward, I_c, I_star, i_ext_vec):
    print(f"I_c = {I_c}")
    print(f"I_star = {I_star}")
    plt.figure(figsize=(7, 3.5))
    plt.plot(i_ext_vec, f_forward, '.k', markersize=15, label='forward')
    plt.plot(i_ext_vec, f_backward, 'ok', markersize=10, markerfacecolor='none',
             linewidth=1, label='backward')
    plt.xlim(i_ext_vec.min(), i_ext_vec.max())
    plt.ylim(0, f_backward.max() * 1.1)
    plt.xlabel(r'$I$ [$\mu$A/cm$^2$]')
    plt.ylabel('$f$ [Hz]')
    plt.axvline(I_star, color='r', linewidth=3)
    plt.axvline(I_c, color='r', linewidth=3)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_rtm_with_autapse_f_i_curve(*simulate_rtm_with_autapse_f_i_curve())